### Setup & Imports

In [ ]:

import pandas as pd
import numpy as np
import torch
import os
import gc
import joblib
import warnings
import gensim.downloader as api
from tqdm import tqdm
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

# Classifiers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# Transformers
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create Base Directories for Checkpointing
os.makedirs("embeddings", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("Directory structure initialized.")

Using device: cuda
Directory structure initialized.


In [4]:
from kaggle_secrets import UserSecretsClient
import os
from huggingface_hub import login

# 1. Access the Kaggle Secrets Vault
try:
    user_secrets = UserSecretsClient()
    my_hf_token = user_secrets.get_secret("HF_TOKEN") # Make sure the name matches your secret exactly
    
    # 2. Inject it into the environment and login
    os.environ["HF_TOKEN"] = my_hf_token
    login(token=my_hf_token)
    print("✅ Hugging Face Token successfully loaded and authenticated!")
    
except Exception as e:
    print(f"⚠️ Could not load HF_TOKEN. Make sure it is attached to the notebook. Error: {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Hugging Face Token successfully loaded and authenticated!


## File Mapping

In [ ]:
print("Mapping Source (Kaggle) and Target (Maze) CSV files...")

# Map each variation to its specific source and target CSV files
variations = {
    "var_1_all_preprocessing": {
        "source": "/kaggle/input/datasets/addysrivats/nlp-generic/variation1_all_preprocessing.csv", 
        "target": "/kaggle/input/datasets/addysrivats/nlp-test/variation1_all_preprocessing.csv"
    },
    "var_2_normalising_casing_space": {
        "source": "/kaggle/input/datasets/addysrivats/nlp-generic/variation2_only_normalising_casing_spaces.csv", 
        "target": "/kaggle/input/datasets/addysrivats/nlp-test/variation2_normalising_casing_spaces.csv"
    },
    "var_3_all_no_emoji_handling": {
        "source": "/kaggle/input/datasets/addysrivats/nlp-generic/variation3_all_preprocessing_except_emoji_handling.csv", 
        "target": "/kaggle/input/datasets/addysrivats/nlp-test/variation3_all_preprocessing_except_no_emoji_handling.csv"
    },
    "var_4_all_no_hash_no_elong": {
        "source": "/kaggle/input/datasets/addysrivats/nlp-generic/variation4_all_preprocessing_except_hashtag_elongation.csv", 
        "target": "/kaggle/input/datasets/addysrivats/nlp-test/variation4_all_preprocessing_except_elongation_hashtag.csv"
    },
    "var_5_all_lowercased": {
        "source": "/kaggle/input/datasets/addysrivats/nlp-generic/variation5_all_preprocessing_and_lowercased.csv", 
        "target": "/kaggle/input/datasets/addysrivats/nlp-test/variation5_all_preprocessing_and_lowercased.csv"
    }
}

# The column names inside your CSVs (Assuming they are named 'CommentText' and 'Sentiment')
TEXT_COL = 'CommentText'
LABEL_COL = 'Sentiment'

print("File mapping complete. Ready for Embedding Factory.")

Mapping Source (Kaggle) and Target (Maze) CSV files...
File mapping complete. Ready for Embedding Factory.


### Load Pre-trained Static Embeddings

In [ ]:
print("Loading Word2Vec, GloVe, and FastText...")
w2v_model = api.load('word2vec-google-news-300')
glove_model = api.load('glove-twitter-200')
fasttext_model = api.load('fasttext-wiki-news-subwords-300')
print("All static dense models loaded successfully!")

###  The Embedding Generators

In [ ]:
# --- STATIC DENSE ENGINE ---
def get_sentence_vector(text, model, vector_size):
    words = str(text).split()
    word_vecs = [model[w] for w in words if w in model]
    if len(word_vecs) == 0: 
        return np.zeros(vector_size)
    return np.mean(word_vecs, axis=0)

def generate_static_embeddings(text_list, model, vector_size):
    return np.array([get_sentence_vector(text, model, vector_size) for text in text_list])

# --- TRANSFORMER ENGINE ---
def get_transformer_embeddings(text_list, model_name, batch_size=64):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModel.from_pretrained(model_name).to(device)
    transformer.eval()
    
    all_embeddings = []
    for i in tqdm(range(0, len(text_list), batch_size), desc=f"Extracting ({model_name})"):
        batch = [str(text) for text in text_list[i : i + batch_size]]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = transformer(**inputs)
        all_embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
        
    del transformer, tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    return np.vstack(all_embeddings)

# --- SBERT ENGINE ---
def get_sbert_embeddings(text_list, model_name='all-mpnet-base-v2'):
    sbert_model = SentenceTransformer(model_name).to(device)
    embeddings = sbert_model.encode([str(text) for text in text_list], batch_size=64, show_progress_bar=True, device=device)
    del sbert_model
    torch.cuda.empty_cache()
    gc.collect()
    return embeddings

###  GENERATE AND SAVE ALL EMBEDDINGS

In [ ]:

for var_name, files in variations.items():
    print(f"\n{'='*50}\nEMBEDDING FACTORY: {var_name}\n{'='*50}")
    
    # 1. Load the specific CSVs for this variation
    source_df = pd.read_csv(files['source']).dropna(subset=[TEXT_COL, LABEL_COL])
    target_df = pd.read_csv(files['target']).dropna(subset=[TEXT_COL, LABEL_COL])
    
    # Create subfolders for this specific variation
    os.makedirs(f"embeddings/{var_name}", exist_ok=True)
    os.makedirs(f"models/{var_name}", exist_ok=True)
    
    # 3. Apply 90/10 Split for Source Data
    X_train_text, X_val_text, y_train, y_val = train_test_split(
        source_df[TEXT_COL].tolist(), source_df[LABEL_COL].values, 
        test_size=0.10, random_state=42, stratify=source_df[LABEL_COL].values
    )
    X_target_text = target_df[TEXT_COL].tolist()
    y_target_true = target_df[LABEL_COL].values # Save target labels for Phase 3
    
    # Save the labels so they align perfectly with the embeddings later
    np.savez_compressed(f"embeddings/{var_name}/labels.npz", train=y_train, val=y_val, target=y_target_true)
    
    def save_embs(emb_name, train_arr, val_arr, test_arr):
        """Helper function to save arrays cleanly"""
        path = f"embeddings/{var_name}/{emb_name}"
        np.savez_compressed(f"{path}_train.npz", embeddings=train_arr)
        np.savez_compressed(f"{path}_val.npz", embeddings=val_arr)
        np.savez_compressed(f"{path}_test.npz", embeddings=test_arr)
        print(f"  -> Saved {emb_name} embeddings successfully.")

    # Generate Embeddings
    print("\n[+] Generating TF-IDF...")
    tfidf = TfidfVectorizer(max_features=5000)
    save_embs("tfidf", tfidf.fit_transform(X_train_text).toarray(), 
                       tfidf.transform(X_val_text).toarray(), 
                       tfidf.transform(X_target_text).toarray())
    
    print("\n[+] Generating Word2Vec...")
    save_embs("w2v", generate_static_embeddings(X_train_text, w2v_model, 300), 
                     generate_static_embeddings(X_val_text, w2v_model, 300), 
                     generate_static_embeddings(X_target_text, w2v_model, 300))
    
    print("\n[+] Generating GloVe...")
    save_embs("glove", generate_static_embeddings(X_train_text, glove_model, 200),
                       generate_static_embeddings(X_val_text, glove_model, 200),
                       generate_static_embeddings(X_target_text, glove_model, 200))
    
    print("\n[+] Generating FastText...")
    save_embs("fasttext", generate_static_embeddings(X_train_text, fasttext_model, 300),
                          generate_static_embeddings(X_val_text, fasttext_model, 300),
                          generate_static_embeddings(X_target_text, fasttext_model, 300))

    print("\n[+] Generating BERT...")
    save_embs("bert", get_transformer_embeddings(X_train_text, 'bert-base-cased'),
                      get_transformer_embeddings(X_val_text, 'bert-base-cased'),
                      get_transformer_embeddings(X_target_text, 'bert-base-cased'))

    print("\n[+] Generating RoBERTa...")
    save_embs("roberta", get_transformer_embeddings(X_train_text, 'roberta-base'),
                         get_transformer_embeddings(X_val_text, 'roberta-base'),
                         get_transformer_embeddings(X_target_text, 'roberta-base'))

    print("\n[+] Generating SBERT...")
    save_embs("sbert", get_sbert_embeddings(X_train_text, 'all-mpnet-base-v2'),
                       get_sbert_embeddings(X_val_text, 'all-mpnet-base-v2'),
                       get_sbert_embeddings(X_target_text, 'all-mpnet-base-v2'))

print("\n\nPHASE 1 COMPLETE: All embeddings generated and secured to disk.")

### TRAINING AND VALIDATION PHASE

In [ ]:

embedding_types = ["tfidf", "w2v", "glove", "fasttext", "bert", "roberta", "sbert"]

ROOT_EMBEDDING_PATH = "/kaggle/input/datasets/addysrivats/embeddings-da/embeddings"

for var_name in variations.keys():
    print(f"\n{'='*50}\nTRAINING PHASE: {var_name}\n{'='*50}")

    os.makedirs(f"models/{var_name}", exist_ok=True)
    
    # Load labels for this variation
    labels = np.load(f"{ROOT_EMBEDDING_PATH}/{var_name}/labels.npz")
    y_train = labels['train']
    y_val = labels['val']
    
    for emb in embedding_types:
        print(f"\n  --- Loading {emb.upper()} Embeddings ---")
        X_train = np.load(f"{ROOT_EMBEDDING_PATH}/{var_name}/{emb}_train.npz")['embeddings']
        X_val = np.load(f"{ROOT_EMBEDDING_PATH}/{var_name}/{emb}_val.npz")['embeddings']
        
        models = {
            "Naive_Bayes": GaussianNB(),
            "LinearSVC": LinearSVC(random_state=42),
            "MLP_NeuralNet": MLPClassifier(hidden_layer_sizes=(128,), max_iter=200, random_state=42),
            "XGBoost": xgb.XGBClassifier(objective='multi:softprob', num_class=3, eval_metric='mlogloss', random_state=42)
        }
        
        for clf_name, clf in models.items():
            # 1. Train
            clf.fit(X_train, y_train)
            
            # 2. Validate
            val_preds = clf.predict(X_val)
            val_acc = accuracy_score(y_val, val_preds)
            print(f"    [Trained] {clf_name.ljust(15)} | Validation Accuracy: {val_acc*100:.2f}%")
            
            # 3. Save Model
            joblib.dump(clf, f"models/{var_name}/{emb}_{clf_name}.pkl")
            
        # Free memory before loading the next embedding matrix
        del X_train, X_val
        gc.collect()

print("\n\nPHASE 2 COMPLETE: All models trained, validated, and saved.")


TRAINING PHASE: var_1_all_preprocessing

  --- Loading TFIDF Embeddings ---
    [Trained] Naive_Bayes     | Validation Accuracy: 52.13%
    [Trained] LinearSVC       | Validation Accuracy: 64.34%
    [Trained] MLP_NeuralNet   | Validation Accuracy: 61.48%
    [Trained] XGBoost         | Validation Accuracy: 61.44%

  --- Loading W2V Embeddings ---
    [Trained] Naive_Bayes     | Validation Accuracy: 49.69%
    [Trained] LinearSVC       | Validation Accuracy: 60.82%
    [Trained] MLP_NeuralNet   | Validation Accuracy: 56.48%
    [Trained] XGBoost         | Validation Accuracy: 60.13%

  --- Loading GLOVE Embeddings ---
    [Trained] Naive_Bayes     | Validation Accuracy: 47.92%
    [Trained] LinearSVC       | Validation Accuracy: 57.25%
    [Trained] MLP_NeuralNet   | Validation Accuracy: 52.88%
    [Trained] XGBoost         | Validation Accuracy: 58.15%

  --- Loading FASTTEXT Embeddings ---
    [Trained] Naive_Bayes     | Validation Accuracy: 48.54%
    [Trained] LinearSVC       | Va

###  DOMAIN ADAPTATION TARGET TESTING (MAZE DATA)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


ROOT_MODELS_PATH="/kaggle/input/datasets/addysrivats/trained-models"


results_tracker = []
misclassifications_tracker = []

embedding_types = ["tfidf", "w2v", "glove", "fasttext", "bert", "roberta", "sbert"]
classifier_names = ["Naive_Bayes", "LinearSVC", "MLP_NeuralNet", "XGBoost"]

print("\nCommencing Zero-Shot Evaluation & Error Logging on Target Domain...")

for var_name, files in variations.items():
    print(f"\nTesting {var_name} Models...")
    
    # 1. Load Ground Truth Labels
    y_target_true = np.load(f"{ROOT_EMBEDDING_PATH}/{var_name}/labels.npz")['target']
    
    # 2. [NEW] Load the raw text to log the misclassifications
    # Using the same error handling we used in Phase 1
    target_df = pd.read_csv(files['target'], encoding='utf-8', encoding_errors='replace')
    target_text_list = target_df[TEXT_COL].astype(str).str.replace(r'[\n\r]+', ' ', regex=True).tolist()
    
    for emb in embedding_types:
        X_target = np.load(f"{ROOT_EMBEDDING_PATH}/{var_name}/{emb}_test.npz")['embeddings']
        
        for clf_name in classifier_names:
            model_path = f"{ROOT_MODELS_PATH}/{var_name}/{emb}_{clf_name}.pkl"
            clf = joblib.load(model_path)
            
            # Predict
            target_preds = clf.predict(X_target)
            
            # --- METRICS CALCULATION ---
            # We use average='weighted' to account for any slight imbalances in the 3 classes
            acc = accuracy_score(y_target_true, target_preds)
            prec = precision_score(y_target_true, target_preds, average='weighted', zero_division=0)
            rec = recall_score(y_target_true, target_preds, average='weighted', zero_division=0)
            f1 = f1_score(y_target_true, target_preds, average='weighted', zero_division=0)
            
            # Log Metrics
            results_tracker.append({
                "Variation": var_name,
                "Embedding": emb.upper(),
                "Classifier": clf_name,
                "Accuracy": round(acc * 100, 2),
                "Precision": round(prec * 100, 2),
                "Recall": round(rec * 100, 2),
                "F1_Score": round(f1 * 100, 2)
            })
            
            # --- ERROR LOGGING ---
            # Find the array indices where the prediction does NOT equal the truth
            error_indices = np.where(target_preds != y_target_true)[0]
            
            for idx in error_indices:
                misclassifications_tracker.append({
                    "Variation": var_name,
                    "Embedding": emb.upper(),
                    "Classifier": clf_name,
                    "True_Label": y_target_true[idx],
                    "Predicted_Label": target_preds[idx],
                    "Comment_Text": target_text_list[idx] # The raw text that confused the model
                })
            
        del X_target
        gc.collect()

print("\nPHASE 3 COMPLETE: Target evaluation and Error Logging finished.")


Commencing Zero-Shot Evaluation & Error Logging on Target Domain...

Testing var_1_all_preprocessing Models...

Testing var_2_normalising_casing_space Models...

Testing var_3_all_no_emoji Models...

Testing var_4_all_no_hash_no_elong Models...

Testing var_5_all_lowercased Models...

PHASE 3 COMPLETE: Target evaluation and Error Logging finished.


In [ ]:
    # 1. Save the Performance Metrics
metrics_df = pd.DataFrame(results_tracker)
metrics_df.to_csv("CIA2_Full_Metrics_Report.csv", index=False)
print("✅ Saved Performance Metrics -> 'CIA2_Full_Metrics_Report.csv'")

# 2. Save the Misclassifications
errors_df = pd.DataFrame(misclassifications_tracker)
errors_df.to_csv("CIA2_Misclassified_Comments_Log.csv", index=False)
print("✅ Saved Error Log -> 'CIA2_Misclassified_Comments_Log.csv'")

✅ Saved Performance Metrics -> 'CIA2_Full_Metrics_Report.csv'
✅ Saved Error Log -> 'CIA2_Misclassified_Comments_Log.csv'


### CHECKING FOR MAX TOKEN LIMIT

In [16]:
/from transformers import AutoTokenizer
import pandas as pd


df = pd.read_csv("/kaggle/input/datasets/addysrivats/yt-comments-48k-clean/youtube_comments_48k_clean.csv")


tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Check how many comments are actually being chopped off
token_counts = [len(tokenizer.encode(text)) for text in df['CommentText']]
clipped = sum(1 for x in token_counts if x > 64)
print(f"Percentage of comments truncated: {clipped / len(df) * 100:.2f}%")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (543 > 512). Running this sequence through the model will result in indexing errors


Percentage of comments truncated: 1.47%


### ENSEMBLING

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import joblib
import gc
import os

ensemble_results = []
BASE_PATH = "/kaggle/input/datasets/addysrivats/embeddings-da/embeddings"

# Ensure the models directory exists to prevent crash on joblib.dump
os.makedirs("models", exist_ok=True)

for var_name in variations.keys():
    print(f"\n{'='*50}\nSTACKING PIPELINE: {var_name}\n{'='*50}")

    # Ensure variation-specific model folder exists
    os.makedirs(f"models/{var_name}", exist_ok=True)

    # 1. Load Ground Truth Labels
    try:
        labels = np.load(f"{BASE_PATH}/{var_name}/labels.npz")
        y_train = labels['train']
        
        y_maze_true = labels['target'] 
    except FileNotFoundError:
        print(f"⚠️ Could not find labels.npz for {var_name} at {BASE_PATH}. Skipping...")
        continue
    except KeyError as e:
        print(f"⚠️ Key Error in labels for {var_name}: {e}. Check if the array key is 'test' instead of 'target'. Skipping...")
        continue

    # 2. Load RoBERTa Embeddings
    print("  -> Loading pre-computed RoBERTa matrices...")
    try:
        X_train = np.load(f"{BASE_PATH}/{var_name}/roberta_train.npz")['embeddings']
        # Assuming your 'roberta_test.npz' contains the Maze data embeddings
        X_maze = np.load(f"{BASE_PATH}/{var_name}/roberta_test.npz")['embeddings'] 
    except Exception as e:
        print(f"⚠️ Error loading embeddings for {var_name}: {e}. Skipping...")
        continue

    # 🛑 3. SANITY CHECK (The Lie Detector)
    print(f"  -> 🔍 VERIFICATION: Training Rows = {X_train.shape[0]} | Maze Rows = {X_maze.shape[0]}")
    if X_maze.shape[0] != len(y_maze_true):
        print(f"⚠️ CRITICAL MISMATCH: Maze embeddings shape {X_maze.shape} does not match label shape {y_maze_true.shape}!")
        continue

    # 4. Define Base Estimators
    estimators = [
        ('svc', LinearSVC(random_state=42, max_iter=2000)), # Added max_iter to prevent convergence warnings
        ('mlp', MLPClassifier(hidden_layer_sizes=(128,), max_iter=500, random_state=42)),
        ('xgb', xgb.XGBClassifier(objective='multi:softprob', num_class=3, eval_metric='mlogloss', random_state=42))
    ]

    # 5. Define the Meta-Learner
    meta_learner = LogisticRegression(random_state=42)

    # 6. Build Stacking Classifier
    stacking_clf = StackingClassifier(
        estimators=estimators,
        final_estimator=meta_learner,
        cv=5,
        n_jobs=-1
    )

    # 7. Train the Ensemble
    print(f"  -> Training Stacking Classifier (Running 5-fold CV on Base Estimators)...")
    stacking_clf.fit(X_train, y_train)

    # 8. Evaluate strictly on the Maze Domain
    print("  -> Evaluating Meta-Learner on Maze Target Domain...")
    maze_preds = stacking_clf.predict(X_maze)
    maze_acc = accuracy_score(y_maze_true, maze_preds)

    print(f"    [Success] Maze Ensemble Accuracy: {maze_acc*100:.2f}%")

    # 9. Log and Save
    ensemble_results.append({
        "Variation": var_name,
        "Model": "RoBERTa_Custom_Stack",
        "Maze_Zero_Shot_Accuracy": round(maze_acc * 100, 2)
    })

    # Save the ensemble model
    joblib.dump(stacking_clf, f"models/{var_name}/roberta_custom_stack.pkl")

    # Flush memory
    del X_train, X_maze, stacking_clf
    gc.collect()

# 10. View Final Leaderboard
if ensemble_results:
    ensemble_df = pd.DataFrame(ensemble_results)
    print("\n🏆 FINAL CUSTOM ENSEMBLE RESULTS ON MAZE DATASET 🏆")
    display(ensemble_df.sort_values(by="Maze_Zero_Shot_Accuracy", ascending=False))

Ensemble Predictions: [1 2 0 1 1]
